In [1]:
import os
import re
import pandas as pd
import numpy as np

In [5]:
BASE_DIR = os.path.abspath("..")
PROCESSED = os.path.join(BASE_DIR, "data", "processed")

print(PROCESSED)
print(os.path.exists(PROCESSED))
print(os.listdir(PROCESSED))

C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\data\processed
True
['knowledge_base.csv', 'master_test.csv', 'master_train.csv', 'master_validation.csv', 'medquad_clean.csv', 'mpib_test_clean.csv', 'mpib_train_clean.csv', 'mpib_validation_clean.csv', 'pubmedqa_clean.csv', 'security_test.csv', 'security_train.csv', 'security_validation.csv']


In [6]:
train = pd.read_csv(os.path.join(PROCESSED, "security_train.csv"))
validation = pd.read_csv(os.path.join(PROCESSED, "security_validation.csv"))
test = pd.read_csv(os.path.join(PROCESSED, "security_test.csv"))

In [8]:
print(train.shape)
print(validation.shape)
print(test.shape)

train.head()

(11463, 6)
(2369, 6)
(2370, 6)


,prompt,attack_type,severity,is_safe,phi_present,source_dataset
0,A 45-year-old male presents to your office wit...,suspicious,2,0,0,MPIB
1,Medical Report of Robert Kumar.,phi,3,0,1,Synthetic-PHI
2,Patient Olivia Kumar is diagnosed with Hyperte...,phi,3,0,1,Synthetic-PHI
3,A 49-year-old man with HIV comes to the physic...,safe,0,1,0,MPIB
4,A 2-week-old female newborn is brought to the ...,safe,0,1,0,MPIB


In [9]:
print(train.isnull().sum())
print(validation.isnull().sum())
print(test.isnull().sum())

prompt            0
attack_type       0
severity          0
is_safe           0
phi_present       0
source_dataset    0
dtype: int64
prompt            0
attack_type       0
severity          0
is_safe           0
phi_present       0
source_dataset    0
dtype: int64
prompt            0
attack_type       0
severity          0
is_safe           0
phi_present       0
source_dataset    0
dtype: int64


In [10]:
train = train.dropna(subset=["prompt"])
validation = validation.dropna(subset=["prompt"])
test = test.dropna(subset=["prompt"])

In [11]:
print("Duplicates Before")

print(train.duplicated(subset=["prompt"]).sum())

train = train.drop_duplicates(subset=["prompt"])

validation = validation.drop_duplicates(subset=["prompt"])

test = test.drop_duplicates(subset=["prompt"])

print("Duplicates After")

print(train.duplicated(subset=["prompt"]).sum())

Duplicates Before
1404
Duplicates After
0


In [13]:
def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text
train["prompt"] = train["prompt"].apply(clean_text)

validation["prompt"] = validation["prompt"].apply(clean_text)

test["prompt"] = test["prompt"].apply(clean_text)    

In [14]:
train["prompt"].head()

0    A 45-year-old male presents to your office wit...
1                      Medical Report of Robert Kumar.
2    Patient Olivia Kumar is diagnosed with Hyperte...
3    A 49-year-old man with HIV comes to the physic...
4    A 2-week-old female newborn is brought to the ...
Name: prompt, dtype: object

In [16]:
label_mapping = {

    "safe":0,

    "malicious":1,

    "phi":2,

    "jailbreak":3,

    "suspicious":4

}
train["label"] = train["attack_type"].map(label_mapping)

validation["label"] = validation["attack_type"].map(label_mapping)

test["label"] = test["attack_type"].map(label_mapping)

In [17]:
train[["attack_type","label"]].head(20)

,attack_type,label
0,suspicious,4
1,phi,2
2,phi,2
3,safe,0
4,safe,0
5,safe,0
6,suspicious,4
7,malicious,1
8,suspicious,4
9,safe,0


In [18]:
train["label"].value_counts()

label
0    5627
1    2874
2     718
4     661
3     179
Name: count, dtype: int64

In [19]:
print(train["label"].unique())

print(validation["label"].unique())

print(test["label"].unique())

[4 2 0 1 3]
[1 3 0 4 2]
[1 0 3 4 2]


In [20]:
train["characters"] = train["prompt"].str.len()

train["words"] = train["prompt"].str.split().str.len()

train[["characters","words"]].describe()

,characters,words
count,10059.000000,10059.000000
mean,611.120489,95.511482
std,470.179010,73.379955
min,2.000000,1.000000
25%,253.000000,35.000000
50%,572.000000,90.000000
75%,868.000000,138.000000
max,3329.000000,496.000000


In [22]:
import os

BASE_DIR = os.path.abspath("..")   # Go from notebooks → project root
PROCESSED = os.path.join(BASE_DIR, "data", "processed")

print(PROCESSED)

C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\data\processed


In [23]:
train.to_csv(
    os.path.join(PROCESSED, "security_train_processed.csv"),
    index=False
)

validation.to_csv(
    os.path.join(PROCESSED, "security_validation_processed.csv"),
    index=False
)

test.to_csv(
    os.path.join(PROCESSED, "security_test_processed.csv"),
    index=False
)

In [24]:
mapping = pd.DataFrame(

    label_mapping.items(),

    columns=[

        "attack_type",

        "label"

    ]

)

mapping.to_csv(

    os.path.join(

        PROCESSED,

        "label_mapping.csv"

    ),

    index=False

)

In [25]:
print("="*60)

print("SECURITY PREPROCESSING COMPLETED")

print("="*60)

print()

print("Train Shape")

print(train.shape)

print()

print("Validation Shape")

print(validation.shape)

print()

print("Test Shape")

print(test.shape)

print()

print("Class Distribution")

print(train["attack_type"].value_counts())

print()

print("Missing Values")

print(train.isnull().sum())

SECURITY PREPROCESSING COMPLETED

Train Shape
(10059, 9)

Validation Shape
(2306, 7)

Test Shape
(2306, 7)

Class Distribution
attack_type
safe          5627
malicious     2874
phi            718
suspicious     661
jailbreak      179
Name: count, dtype: int64

Missing Values
prompt            0
attack_type       0
severity          0
is_safe           0
phi_present       0
source_dataset    0
label             0
characters        0
words             0
dtype: int64
